In [0]:
df_wb_populacao_bronze = spark.read.table("portfolio_energia.bronze.world_bank_bronze_populacao")
df_wb_fdi_bronze = spark.read.table("portfolio_energia.bronze.world_bank_bronze_fdi")
df_wb_eletricidade_bronze = spark.read.table("portfolio_energia.bronze.world_bank_bronze_eletricidade")
df_wb_pib_bronze = spark.read.table("portfolio_energia.bronze.world_bank_bronze_pib_per_capita")

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

def silver_treatment_wb(df_start):
    janela_deduplicacao = Window.partitionBy("iso3", "date").orderBy(col("data_ingestao_bronze").desc())
    
    df_final = (
        df_start
        .dropna("any", subset=["iso3", "date"])
        .withColumn("numero_linha", row_number().over(janela_deduplicacao))
        .filter(col("numero_linha") == 1)
        .drop("numero_linha")
        .withColumn("country_name", trim(col("country_name")))
        .withColumn("iso3", upper(trim(col("iso3"))))
        .withColumn("metric_name", trim(col("metric_name")))
        .withColumn("date", to_date(col("date"), "yyyy"))
        .withColumn("metric", col("metric").cast("double"))
        .withColumn("ingestion_date", current_timestamp())
        .fillna(-1.0, subset=["metric"])
    )
    return df_final

df_wb_populacao_silver = silver_treatment_wb(df_wb_populacao_bronze)
df_wb_fdi_silver = silver_treatment_wb(df_wb_fdi_bronze)
df_wb_eletricidade_silver = silver_treatment_wb(df_wb_eletricidade_bronze)
df_wb_pib_silver = silver_treatment_wb(df_wb_pib_bronze)

def salvar_silver_incremental_wb(df_tratado, nome_tabela_silver):
    caminho_tabela = f"portfolio_energia.silver.{nome_tabela_silver}"
    
    if not spark.catalog.tableExists(caminho_tabela):
        print(f"Criando tabela {caminho_tabela} pela primeira vez...")
        df_tratado.write.format("delta").saveAsTable(caminho_tabela)
        
    else:
        df_tratado.createOrReplaceTempView("vw_dados_novos_wb")
        
        spark.sql(f"""
            MERGE INTO {caminho_tabela} AS destino
            USING vw_dados_novos_wb AS origem
            ON destino.iso3 = origem.iso3 AND destino.date = origem.date
            
            WHEN MATCHED THEN UPDATE SET *
            WHEN NOT MATCHED THEN INSERT *
        """)

salvar_silver_incremental_wb(df_wb_populacao_silver, "world_bank_silver_populacao")
salvar_silver_incremental_wb(df_wb_fdi_silver, "world_bank_silver_pfdi")
salvar_silver_incremental_wb(df_wb_eletricidade_silver, "world_bank_silver_peletricidade")
salvar_silver_incremental_wb(df_wb_pib_silver, "world_bank_silver_pib_per_capita")